In [ ]:
import pandas as pd
import numpy as np
import joblib
import re

import nltk
from nltk.tokenize import word_tokenize
from gensim.models import Word2Vec

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC

from sklearn import preprocessing
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import pickle
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from keras.layers import Dense, Embedding, GlobalAveragePooling1D, LSTM, Input, Dropout, Bidirectional
from tensorflow.keras.metrics import Precision, Recall 
from imblearn.over_sampling import RandomOverSampler
from tensorflow.keras.models import load_model

In [ ]:
df = pd.read_csv("data_clean.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   Unnamed: 0            350 non-null    int64 
 1   title                 350 non-null    object
 2   name                  350 non-null    object
 3   stars                 350 non-null    int64 
 4   text                  350 non-null    object
 5   label_sentimen        350 non-null    object
 6   text_clean            347 non-null    object
 7   text_casefoldingText  347 non-null    object
 8   text_slangwords       347 non-null    object
 9   text_stemming         347 non-null    object
 10  text_tokenizingText   350 non-null    object
 11  text_stopword         350 non-null    object
 12  text_final            346 non-null    object
dtypes: int64(2), object(11)
memory usage: 35.7+ KB


In [ ]:
df.dropna(subset=['text_final'], inplace=True)
df = df[df['text_final'].str.strip() != '']

In [ ]:
df[['label_sentimen']].value_counts()

label_sentimen
Positif           308
Negatif            38
Name: count, dtype: int64

In [ ]:
label_encoder = preprocessing.LabelEncoder() 
df['sentimen_encode'] = label_encoder.fit_transform(df['label_sentimen'])

In [ ]:
X_input = df['text_final']
y_input = df['sentimen_encode']

In [ ]:
tfidf = TfidfVectorizer(max_features=5000, min_df=2, ngram_range=(1,2))
X_train, X_test, y_train, y_test = train_test_split(X_input, y_input, test_size=0.2, random_state=42)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)
ros = RandomOverSampler(random_state=42)
X_train_tfidf, y_train_tfidf = ros.fit_resample(X_train_tfidf, y_train)

In [ ]:
random_forest = RandomForestClassifier(
    n_estimators=200,
    min_samples_split=10,
    min_samples_leaf=4,
    random_state=42,
    max_features='sqrt',
    class_weight='balanced',
)

random_forest.fit(X_train_tfidf.toarray(), y_train_tfidf)

RandomForestClassifier(class_weight='balanced', min_samples_leaf=4,
                       min_samples_split=10, n_estimators=200, random_state=42)

In [ ]:
y_pred_train_tfidf = random_forest.predict(X_train_tfidf.toarray())
y_pred_test_tfidf = random_forest.predict(X_test_tfidf.toarray())
 
accuracy_train_tfidf = accuracy_score(y_pred_train_tfidf, y_train_tfidf)
accuracy_test_tfidf = accuracy_score(y_pred_test_tfidf, y_test_tfidf)

print('accuracy_train:', accuracy_train_tfidf)
print('accuracy_test:', accuracy_test_tfidf)
print(classification_report(y_test_tfidf, y_pred_test_tfidf))
print(confusion_matrix(y_test_tfidf, y_pred_test_tfidf))

accuracy_train: 0.9044715447154471
accuracy_test: 0.8548387096774194
              precision    recall  f1-score   support

           0       0.89      0.76      0.82        54
           1       0.83      0.93      0.88        70

    accuracy                           0.85       124
   macro avg       0.86      0.84      0.85       124
weighted avg       0.86      0.85      0.85       124

[[41 13]
 [ 5 65]]


In [ ]:
joblib.dump(tfidf, './assets/tf_idf.joblib')
joblib.dump(random_forest, './assets/rf_tfidf.joblib')

['./assets/rf_tfidf.joblib']

In [ ]:
loaded_tfidf = joblib.load('./assets/tf_idf.joblib')
loaded_rf = joblib.load('./assets/rf_tfidf.joblib')

label_map = {0: 'NEGATIVE', 1: 'POSITIVE'}

import re, string
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from nltk.corpus import stopwords

stemmer = StemmerFactory().create_stemmer()
listStopwords = set(stopwords.words('indonesian'))
listStopwords.update(stopwords.words('english'))
listStopwords -= {'nggak', 'ngga', 'gak', 'ga', 'gk', 'tidak', 'tdk', 'tak'}

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in listStopwords]
    tokens = [stemmer.stem(w) for w in tokens]
    return ' '.join(tokens)

def predict(text):
    clean_input = clean_text(text)
    vec_input = loaded_tfidf.transform([clean_input])
    
    pred_index = loaded_rf.predict(vec_input)[0]
    pred_proba = loaded_rf.predict_proba(vec_input)[0]
    
    result = label_map[pred_index]
    score = pred_proba[pred_index] * 100
    
    return result, score

txt = "Sangat mengecewakan, kotor, jorok, rusak, panas tidak ada AC"
result, score = predict(txt)
print(f"Analyzed: {result} (Score: {score:.1f}%)")

Analyzed: NEGATIVE (Score: 73.3%)


In [ ]:
svm = LinearSVC(random_state=42, C=1.0)

svm.fit(X_train_tfidf, y_train_tfidf)

y_train_svm = svm.predict(X_train_tfidf)
y_pred_svm = svm.predict(X_test_tfidf)

acc_train_svm = accuracy_score(y_train_tfidf, y_train_svm)
acc_test_svm = accuracy_score(y_test_tfidf, y_pred_svm)

print(f"train accuracy score: {acc_train_svm}")
print(f"test accuracy_score: {acc_test_svm}")
print(classification_report(y_test_tfidf, y_pred_svm))
print(confusion_matrix(y_test_tfidf, y_pred_svm))

joblib.dump(svm, './assets/svm_tfidf.joblib')

train accuracy score: 0.9532520325203252
test accuracy_score: 0.8870967741935484
              precision    recall  f1-score   support

           0       0.90      0.83      0.87        54
           1       0.88      0.93      0.90        70

    accuracy                           0.89       124
   macro avg       0.89      0.88      0.88       124
weighted avg       0.89      0.89      0.89       124

[[45  9]
 [ 5 65]]


['./assets/svm_tfidf.joblib']

In [ ]:
loaded_svm = joblib.load('./assets/svm_tfidf.joblib')

def predict_svm_tfidf(text):
    clean_input = clean_text(text)
    vec_input = loaded_tfidf.transform([clean_input])
    
    pred_index = loaded_svm.predict(vec_input)[0]
    result = label_map[pred_index]
    
    decision_function = loaded_svm.decision_function(vec_input)[0]
    
    if hasattr(decision_function, '__len__'):
        score_mentah = decision_function[pred_index]
    else:
        score_mentah = float(decision_function)
    
    return result, score_mentah

txt = "Sangat mengecewakan, kotor, jorok, rusak, panas tidak ada AC"
result, score = predict_svm_tfidf(txt)

print(f"Analyzed: {result} (Confidence: {score:.2f})")

Analyzed: NEGATIVE (Confidence: -1.43)


### **Model Naive Bayes & TF-IDF**

In [ ]:
from sklearn.naive_bayes import MultinomialNB

# Training
naive_bayes = MultinomialNB()
naive_bayes.fit(X_train_tfidf, y_train_tfidf)

# Evaluasi
y_pred_train_nb = naive_bayes.predict(X_train_tfidf)
y_pred_test_nb  = naive_bayes.predict(X_test_tfidf)

accuracy_train_nb = accuracy_score(y_train_tfidf, y_pred_train_nb)
accuracy_test_nb  = accuracy_score(y_test_tfidf,  y_pred_test_nb)

print('accuracy_train:', accuracy_train_nb)
print('accuracy_test: ', accuracy_test_nb)
print(classification_report(y_test_tfidf, y_pred_test_nb))
print(confusion_matrix(y_test_tfidf, y_pred_test_nb))

joblib.dump(naive_bayes, './assets/nb_tfidf.joblib')

accuracy_train: 0.9308943089430894
accuracy_test:  0.8870967741935484
              precision    recall  f1-score   support

           0       0.84      0.91      0.88        54
           1       0.92      0.87      0.90        70

    accuracy                           0.89       124
   macro avg       0.88      0.89      0.89       124
weighted avg       0.89      0.89      0.89       124

[[49  5]
 [ 9 61]]


['./assets/nb_tfidf.joblib']

In [ ]:
loaded_nb = joblib.load('./assets/nb_tfidf.joblib')

def predict_nb_tfidf(text):
    clean_input = clean_text(text)
    vec_input = loaded_tfidf.transform([clean_input])

    pred_index = loaded_nb.predict(vec_input)[0]
    pred_proba = loaded_nb.predict_proba(vec_input)[0]

    result = label_map[pred_index]
    score  = pred_proba[pred_index] * 100

    return result, score

txt = "Sangat mengecewakan, kotor, jorok, rusak, panas tidak ada AC"
result, score = predict_nb_tfidf(txt)
print(f"Analyzed: {result} (Score: {score:.1f}%)")

Analyzed: NEGATIVE (Score: 96.7%)
